# ☀️ MAGFiLO V9 — Evaluation-Aligned Solar Filament Segmentation

**Competition:** [Solar Filament Segmentation Challenge 2026](https://www.kaggle.com/competitions/filament-segmentation-2026) (Kaggle / IEEE BigData Cup 2026), sponsored by the NSF National Solar Observatory (NSO).

## Problem
Solar filaments are elongated clouds of relatively cool, dense plasma suspended above the Sun's surface by magnetic field lines, visible in H-alpha observations. They are precursors to solar eruptive events such as Coronal Mass Ejections (CMEs), which can disrupt satellite communication, GPS, and power grids — making automated, precise filament segmentation directly relevant to space weather forecasting.

This notebook segments individual solar filaments from full-disk GONG H-alpha images (2048×2048 px), producing a separate instance mask for each filament, including fine-scale structures ("barbs" — thin protrusions extending from the filament spine).

## Data
**MAGFiLO** (Manually Annotated GONG Filaments in H-alpha Observations) — the ground-truth dataset for this competition, providing COCO-style instance segmentation annotations (masks, bounding boxes, spines) curated by multiple human annotators.
> Ahmadzadeh, A., Adhyapak, R., Chaurasiya, K., et al. (2024). *A dataset of manually annotated filaments from H-alpha observations.* Scientific Data, 11, 1031. https://doi.org/10.1038/s41597-024-03876-y

## Evaluation
As of the competition's Aug 7, 2026 update, submissions are scored primarily by the **Panoptic Quality (PQ)** metric rather than pixel-wise Dice, jointly measuring detection quality (are filaments correctly found, penalizing false positives/negatives) and segmentation quality (IoU of correctly matched instances).
> Kirillov, A., He, K., Girshick, R., Rother, C., & Dollár, P. (2019). *Panoptic Segmentation.* CVPR 2019. https://doi.org/10.1109/CVPR.2019.00963

## Our Approach
A two-stage **detect-then-segment** pipeline: (1) a YOLO detector locates each filament as a bounding box, (2) a crop-level segmentation model predicts a precise per-filament mask within each detected region, at near-native resolution to preserve fine barb structures — avoiding both the ambiguous instance-separation of whole-image + connected-components approaches and the resolution loss of downsampling a full 2048×2048 image.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import subprocess
print(subprocess.run(['find', '/kaggle/input', '-name', '*.pt'], capture_output=True, text=True).stdout)
print(subprocess.run(['find', '/kaggle/input', '-name', '*.pth'], capture_output=True, text=True).stdout)

In [ ]:
yolo_axis = YOLO('/kaggle/input/notebooks/muhammadshoaibaltaf/magfilo-v9-evaluation-aligned-solar-filament-s/best_crop_model.pth')
yolo_obb = YOLO('/kaggle/input/notebooks/muhammadshoaibaltaf/obb-with-rotated-crop-segmentation/best_crop_model.pth')

crop_model_unet = smp.UnetPlusPlus(encoder_name='efficientnet-b4', encoder_weights=None, in_channels=1, classes=1, activation=None).to(DEVICE)
crop_model_unet.load_state_dict(torch.load('/kaggle/input/notebooks/muhammadshoaibaltaf/magfilo-v9-evaluation-aligned-solar-filament-s/yolo_runs/filament_detector/weights/best.pt
', map_location=DEVICE))
crop_model_unet.eval()

crop_model_edge = EdgeAttNet(in_channels=1, out_channels=1, base_channels=64, num_heads=4).to(DEVICE)
crop_model_edge.load_state_dict(torch.load('<real EdgeAttNet crop weights path>', map_location=DEVICE))
crop_model_edge.eval()

In [ ]:
import os
print(os.listdir('/kaggle/working'))

In [ ]:
!pip install segmentation-models-pytorch

## 1. Setup, Data Loading, and Train/Validation Split

We load MAGFiLO's COCO-style annotations via `pycocotools`.

**Important dataset detail:** MAGFiLO includes cases where multiple human annotators independently labeled the *same physical observation*; each annotator's labels appear under a distinct `image_id` (`<annotator>-<observation_id>`) sharing the same `file_name`. To prevent train/validation leakage, we split on the unique **photo** (`file_name`) first, then expand back out to all annotator-`image_id`s belonging to each photo.

We also define the segmentation loss here: a combined BCE + Tversky loss, weighting false negatives more heavily than false positives (β > α) to specifically target under-segmentation of thin barb structures.
> Salehi, S. S. M., Erdogmus, D., & Gholipour, A. (2017). *Tversky loss function for image segmentation using 3D fully convolutional deep networks.* MLMI 2017. https://doi.org/10.1007/978-3-319-67389-9_44

In [ ]:

import os
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils
from scipy import ndimage
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

TRAIN_JSON = '/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'
TRAIN_IMG_DIR = '/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/train_images'
TEST_IMG_DIR = '/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images'
IMAGE_SIZE = 512
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

coco = COCO(TRAIN_JSON)
image_ids = list(coco.imgs.keys())
photo_to_image_ids = {}
for image_id in image_ids:
    file_name = coco.loadImgs([image_id])[0]['file_name']
    photo_to_image_ids.setdefault(file_name, []).append(image_id)

unique_photos = list(photo_to_image_ids.keys())
train_photos, val_photos = train_test_split(unique_photos, test_size=0.15, random_state=SEED)

train_ids = [iid for photo in train_photos for iid in photo_to_image_ids[photo]]
val_ids = [iid for photo in val_photos for iid in photo_to_image_ids[photo]]
test_image_paths = sorted(os.path.join(TEST_IMG_DIR, f) for f in os.listdir(TEST_IMG_DIR))

print(f"Unique photos: {len(unique_photos)} | Train image_ids: {len(train_ids)} | Val image_ids: {len(val_ids)}")

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        # beta > alpha = penalize missed pixels (false negatives) harder than extra pixels
        # this directly targets thin barbs getting missed
        super().__init__()
        self.alpha, self.beta, self.smooth = alpha, beta, smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)
        tp = (probs * targets).sum()
        fp = (probs * (1 - targets)).sum()
        fn = ((1 - probs) * targets).sum()
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return 1 - tversky

class BCETverskyLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.tversky = TverskyLoss()

    def forward(self, logits, targets):
        return 0.5 * self.bce(logits, targets) + 0.5 * self.tversky(logits, targets)




def dice_score(logits, targets, threshold=0.5, smooth=1e-6):
    preds = (torch.sigmoid(logits) > threshold).float().view(-1); t = targets.view(-1)
    return (2. * (preds * t).sum() + smooth) / (preds.sum() + t.sum() + smooth)

def mask_to_rle(single_mask):
    single_mask = np.asfortranarray(single_mask.astype('uint8'))
    return mask_utils.encode(single_mask)['counts'].decode('utf-8')

## 2. Stage 1 — Object Detection Data Preparation

We convert COCO bounding boxes into YOLO-format labels for training a filament detector, writing one label file per unique photo (not per annotator-`image_id`) to avoid duplicate/conflicting boxes for the same filament confusing the detector.

In [ ]:
import shutil
shutil.rmtree('/kaggle/working/yolo_data', ignore_errors=True)

In [ ]:
YOLO_DATA_DIR = '/kaggle/working/yolo_data'
for split in ['train', 'val']:
    os.makedirs(f'{YOLO_DATA_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{YOLO_DATA_DIR}/labels/{split}', exist_ok=True)

def convert_to_yolo_format(image_ids, split):
    seen_files = set()  # only write labels once per unique photo — stops multi-annotator duplication
    for image_id in image_ids:
        img_info = coco.loadImgs([image_id])[0]
        file_name = img_info['file_name']
        if file_name in seen_files:
            continue
        seen_files.add(file_name)

        img_w, img_h = img_info['width'], img_info['height']
        src_path = os.path.join(TRAIN_IMG_DIR, file_name)
        dst_path = os.path.join(YOLO_DATA_DIR, 'images', split, file_name)
        if not os.path.exists(dst_path):
            shutil.copy(src_path, dst_path)

        ann_ids = coco.getAnnIds(imgIds=[image_id])
        anns = coco.loadAnns(ann_ids)

        label_name = os.path.splitext(file_name)[0] + '.txt'
        label_path = os.path.join(YOLO_DATA_DIR, 'labels', split, label_name)

        with open(label_path, 'w') as f:   # 'w' — safe since each file only hit once
            for ann in anns:
                x_min, y_min, box_w, box_h = ann['bbox']
                x_center = (x_min + box_w / 2) / img_w
                y_center = (y_min + box_h / 2) / img_h
                f.write(f"0 {x_center:.6f} {y_center:.6f} {box_w/img_w:.6f} {box_h/img_h:.6f}\n")

convert_to_yolo_format(train_ids, 'train')
convert_to_yolo_format(val_ids, 'val')
print("YOLO-format dataset built.")

In [ ]:
yaml_content = f"""path: {YOLO_DATA_DIR}
train: images/train
val: images/val

names:
  0: filament
"""
with open(f'{YOLO_DATA_DIR}/data.yaml', 'w') as f:
    f.write(yaml_content)
print(open(f'{YOLO_DATA_DIR}/data.yaml').read())    

In [ ]:
print(len(train_ids), len(val_ids))

## 3. Stage 1 — Training the Filament Detector

We fine-tune **YOLO11s** (Ultralytics), initialized from COCO-pretrained weights, at `imgsz=1536` — higher than the segmentation stage's resolution, since filaments are small relative to the 2048×2048 frame and benefit from extra input resolution for reliable localization.
> Ultralytics YOLO11. https://docs.ultralytics.com/models/yolo11/

In [ ]:
# Add this near the top of notebook, before heavy training — helps reduce memory fragmentation
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
import gc
print(f"GPU memory in use before cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# release anything not needed for crop training specifically
for var_name in ['yolo_model', 'pred_df', 'pred_df_no_crf', 'pred_df_crf', 'overlap_df', 'overlap_df_no_crf', 'overlap_df_crf']:
    if var_name in dir():
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory in use after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
!pip install -U ultralytics -q
from ultralytics import YOLO

yolo_model = YOLO('yolo26l.pt')

train_results = yolo_model.train(
    data=f'{YOLO_DATA_DIR}/data.yaml',
    optimizer='AdamW',
    lr0=0.00092,
    warmup_epochs=5,
    iou=0.5,
    epochs=50,
    imgsz=1280,
    batch=8,
    patience=8,
    project='/kaggle/working/yolo_runs',
    name='filament_detector',
    pretrained=True,
)



YOLO_WEIGHTS = str(train_results.save_dir / 'weights' / 'best.pt')
print("Weights saved at:", YOLO_WEIGHTS)

In [ ]:
yolo_metrics = yolo_model.val(data=f'{YOLO_DATA_DIR}/data.yaml')
print(f"mAP50: {yolo_metrics.box.map50:.4f} | mAP50-95: {yolo_metrics.box.map:.4f} | Precision: {yolo_metrics.box.mp:.4f} | Recall: {yolo_metrics.box.mr:.4f}")

In [ ]:
torch.cuda.empty_cache()  

## 4. Stage 2 — Crop-Level Segmentation

For each ground-truth filament, we extract a padded crop around its bounding box and train a model to predict that filament's precise mask within the crop. Padding (`PAD_RATIO`) gives surrounding context beyond the tight box.

**Model:** UNet++ with an EfficientNet-B4 encoder pretrained on ImageNet, via `segmentation_models_pytorch`.
> Zhou, Z., Rahman Siddiquee, M. M., Tajbakhsh, N., & Liang, J. (2018). *UNet++: A Nested U-Net Architecture for Medical Image Segmentation.* DLMIA 2018. https://doi.org/10.1007/978-3-030-00889-5_1
> Tan, M., & Le, Q. (2019). *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks.* ICML 2019.
> Iakubovskii, P. *Segmentation Models PyTorch.* https://github.com/qubvel-org/segmentation_models.pytorch

In [ ]:
CROP_SIZE = 512
PAD_RATIO = 0.2

def build_annotation_list(image_ids):
    ann_list = []
    for image_id in image_ids:
        ann_ids = coco.getAnnIds(imgIds=[image_id])
        for ann_id in ann_ids:
            ann_list.append((image_id, ann_id))
    return ann_list

train_ann_list = build_annotation_list(train_ids)
val_ann_list = build_annotation_list(val_ids)
print(f"Train filaments: {len(train_ann_list)} | Val filaments: {len(val_ann_list)}")

In [ ]:
class FilamentCropDataset(Dataset):
    def __init__(self, coco, ann_list, img_dir, crop_size=CROP_SIZE, pad_ratio=PAD_RATIO, transform=None):
        self.coco = coco
        self.ann_list = ann_list
        self.img_dir = img_dir
        self.crop_size = crop_size
        self.pad_ratio = pad_ratio
        self.transform = transform
        self._img_cache = {}

    def __len__(self):
        return len(self.ann_list)

    def _load_image(self, image_id):
        if image_id not in self._img_cache:
            img_info = self.coco.loadImgs([image_id])[0]
            img_path = os.path.join(self.img_dir, img_info['file_name'])
            self._img_cache[image_id] = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        return self._img_cache[image_id]

    def __getitem__(self, idx):
        image_id, ann_id = self.ann_list[idx]
        ann = self.coco.loadAnns([ann_id])[0]

        img = self._load_image(image_id)
        h, w = img.shape
        full_mask = self.coco.annToMask(ann)

        x_min, y_min, box_w, box_h = ann['bbox']
        pad_w, pad_h = box_w * self.pad_ratio, box_h * self.pad_ratio
        x1 = max(0, int(x_min - pad_w)); y1 = max(0, int(y_min - pad_h))
        x2 = min(w, int(x_min + box_w + pad_w)); y2 = min(h, int(y_min + box_h + pad_h))

        img_crop = cv2.resize(img[y1:y2, x1:x2], (self.crop_size, self.crop_size))
        mask_crop = cv2.resize(full_mask[y1:y2, x1:x2], (self.crop_size, self.crop_size), interpolation=cv2.INTER_NEAREST)

        img_crop = np.expand_dims(img_crop, axis=-1)
        mask_crop = mask_crop.astype('float32')

        augmented = self.transform(image=img_crop, mask=mask_crop)
        return augmented['image'], augmented['mask'].unsqueeze(0).float()

In [ ]:
crop_train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.CLAHE(clip_limit=2.0, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.5, contrast_limit=0.5, p=0.5),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])
crop_val_transform = A.Compose([
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

train_crop_dataset = FilamentCropDataset(coco, train_ann_list, TRAIN_IMG_DIR, transform=crop_train_transform)
val_crop_dataset = FilamentCropDataset(coco, val_ann_list, TRAIN_IMG_DIR, transform=crop_val_transform)

train_crop_loader = DataLoader(train_crop_dataset, batch_size=8, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_crop_loader = DataLoader(val_crop_dataset, batch_size=8, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

print(f"Train crops: {len(train_crop_dataset)} | Val crops: {len(val_crop_dataset)}")

In [ ]:
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)
class EGMHSA(nn.Module):
    def __init__(self, channels, num_heads=4, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=channels, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(channels)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x_flat, edge_bias):
        q = x_flat + edge_bias; k = x_flat + edge_bias; v = x_flat
        attn_out, _ = self.mha(q, k, v)
        return self.norm(x_flat + self.dropout(attn_out))
class EdgeAttNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=64, num_heads=4):
        super().__init__()
        c1, c2, c3, c4 = base_channels, base_channels*2, base_channels*4, base_channels*8
        self.enc1, self.enc2, self.enc3, self.enc4 = DoubleConv(in_channels, c1), DoubleConv(c1, c2), DoubleConv(c2, c3), DoubleConv(c3, c4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck_conv = DoubleConv(c4, c4)
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        self.edge_proj = nn.Conv2d(1, c4, kernel_size=1)
        self.eg_mhsa1, self.eg_mhsa2 = EGMHSA(c4, num_heads), EGMHSA(c4, num_heads)
        self.up1, self.dec1 = nn.ConvTranspose2d(c4, c3, 2, stride=2), DoubleConv(c4 + c3, c3)
        self.up2, self.dec2 = nn.ConvTranspose2d(c3, c2, 2, stride=2), DoubleConv(c3 + c2, c2)
        self.up3, self.dec3 = nn.ConvTranspose2d(c2, c1, 2, stride=2), DoubleConv(c2 + c1, c1)
        self.up4, self.dec4 = nn.ConvTranspose2d(c1, c1, 2, stride=2), DoubleConv(c1 + c1, c1)
        self.final_conv = nn.Conv2d(c1, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        x4 = self.enc4(self.pool(x3))
        z = self.bottleneck_conv(self.pool(x4))

        B, C, Hb, Wb = z.shape
        edge = torch.sigmoid(self.edge_conv(x))
        edge = F.interpolate(edge, size=(Hb, Wb), mode='bilinear', align_corners=False)
        edge = self.edge_proj(edge)
        edge_flat = edge.flatten(2).transpose(1, 2)

        z_flat = z.flatten(2).transpose(1, 2)
        z_flat = self.eg_mhsa1(z_flat, edge_flat)
        z_flat = self.eg_mhsa2(z_flat, edge_flat)
        z = z_flat.transpose(1, 2).view(B, C, Hb, Wb)
        d = self.dec1(torch.cat([self.up1(z), x4], dim=1))
        d = self.dec2(torch.cat([self.up2(d), x3], dim=1))
        d = self.dec3(torch.cat([self.up3(d), x2], dim=1))
        d = self.dec4(torch.cat([self.up4(d), x1], dim=1))
        return self.final_conv(d)


class EdgeAttNetLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs, t = torch.sigmoid(logits).view(-1), targets.view(-1)
        dice_loss = 1 - (2*(probs*t).sum()+self.smooth)/(probs.sum()+t.sum()+self.smooth)
        return self.bce(logits, targets) + dice_loss


crop_model = EdgeAttNet(in_channels=1, out_channels=1, base_channels=64, num_heads=4).to(DEVICE)
crop_criterion = EdgeAttNetLoss()
crop_optimizer = torch.optim.Adam(crop_model.parameters(), lr=1e-4)   # plain Adam, no pretrained weights — trained fully from scratch

CROP_NUM_EPOCHS, CROP_PATIENCE = 30, 8   # bumped from 12 — no pretrained init this time, needs more runway to converge
crop_scheduler = CosineAnnealingLR(crop_optimizer, T_max=CROP_NUM_EPOCHS)

In [ ]:
crop_model = smp.UnetPlusPlus(
    encoder_name='efficientnet-b4',
    encoder_weights='imagenet',
    in_channels=1,
    classes=1,
    activation=None,
).to(DEVICE)
crop_criterion = BCETverskyLoss()
crop_optimizer = torch.optim.AdamW(crop_model.parameters(), lr=2e-4, weight_decay=5e-5)

CROP_NUM_EPOCHS, CROP_PATIENCE = 12, 7   # your best epoch has landed between 12-20 across every run so far — 22 gives headroom without an 8-epoch run's risk of cutting off early
crop_scheduler = CosineAnnealingLR(crop_optimizer, T_max=CROP_NUM_EPOCHS)   # matches CROP_NUM_EPOCHS so the cosine decay actually completes

In [ ]:
torch.cuda.empty_cache()  

In [ ]:
best_crop_val_dice, crop_epochs_no_improve = 0.0, 0
scaler = torch.amp.GradScaler('cuda')
torch.backends.cudnn.benchmark = True

for epoch in range(CROP_NUM_EPOCHS):
    crop_model.train()
    train_loss = 0.0
    for imgs, masks in train_crop_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        crop_optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            logits = crop_model(imgs)
            loss = crop_criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(crop_optimizer)
        scaler.update()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_crop_dataset)

    crop_model.eval()
    val_loss, val_dice = 0.0, 0.0
    with torch.no_grad():
        for imgs, masks in val_crop_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = crop_model(imgs)
            val_loss += crop_criterion(logits, masks).item() * imgs.size(0)
            val_dice += dice_score(logits, masks).item() * imgs.size(0)
    val_loss /= len(val_crop_dataset); val_dice /= len(val_crop_dataset)
    crop_scheduler.step()

    print(f"Epoch {epoch+1}/{CROP_NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}")

    if val_dice > best_crop_val_dice:
        best_crop_val_dice = val_dice; crop_epochs_no_improve = 0
        torch.save(crop_model.state_dict(), 'best_crop_model.pth')
        print(f"  -> New best crop model saved (Dice: {best_crop_val_dice:.4f})")
    else:
        crop_epochs_no_improve += 1
        if crop_epochs_no_improve >= CROP_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}"); break

print(f"\nBest crop-level validation Dice: {best_crop_val_dice:.4f}")

## 5. Inference — Combining Detection and Segmentation

We run the trained detector on each test image, then the segmentation model on each detected region. Since detections can spatially overlap (especially near touching filaments), and the competition explicitly disallows overlapping masks for the same image, we resolve conflicts by processing detections in descending confidence order and assigning each pixel to the first (highest-confidence) filament that claims it.

In [ ]:
yolo_model = YOLO(YOLO_WEIGHTS)
crop_model.load_state_dict(torch.load('best_crop_model.pth', map_location=DEVICE))
crop_model.eval()

In [ ]:
def predict_filaments_for_image(image_path, conf_threshold=0.25, pad_ratio=PAD_RATIO, crop_size=CROP_SIZE, mask_threshold=0.5):
    img_full = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    h, w = img_full.shape

    yolo_pred = yolo_model.predict(image_path, imgsz=1280, conf=conf_threshold, verbose=False)[0]
    boxes = yolo_pred.boxes.xyxy.cpu().numpy()
    confs = yolo_pred.boxes.conf.cpu().numpy()

    # process highest-confidence detections first — they get first claim on any contested pixels
    order = np.argsort(-confs)
    boxes = boxes[order]

    occupied = np.zeros((h, w), dtype=bool)  # tracks pixels already claimed by a higher-confidence filament
    instance_masks = []

    for x1, y1, x2, y2 in boxes:
        box_w, box_h = x2 - x1, y2 - y1
        pad_w, pad_h = box_w * pad_ratio, box_h * pad_ratio
        cx1, cy1 = max(0, int(x1 - pad_w)), max(0, int(y1 - pad_h))
        cx2, cy2 = min(w, int(x2 + pad_w)), min(h, int(y2 + pad_h))

        crop = img_full[cy1:cy2, cx1:cx2]
        if crop.size == 0:
            continue
        crop_norm = (cv2.resize(crop, (crop_size, crop_size)).astype('float32') / 255.0 - 0.5) / 0.5
        crop_tensor = torch.from_numpy(crop_norm).unsqueeze(0).unsqueeze(0).float().to(DEVICE)

        with torch.no_grad():
            probs = torch.sigmoid(crop_model(crop_tensor)).squeeze().cpu().numpy()

        probs_resized = cv2.resize(probs, (cx2 - cx1, cy2 - cy1))
        binary_crop = (probs_resized > mask_threshold).astype('uint8')

        full_mask = np.zeros((h, w), dtype='uint8')
        full_mask[cy1:cy2, cx1:cx2] = binary_crop

        full_mask[occupied] = 0   # strip out any pixel already claimed by a higher-confidence detection

        if full_mask.sum() == 0:
            continue   # this detection's whole region got absorbed by a stronger neighbor — skip it

        occupied |= full_mask.astype(bool)
        instance_masks.append(full_mask)

    return instance_masks

## 7. Local Validation — Panoptic Quality (Official Metric)

The helper functions below are adapted directly from the competition organizers' published self-evaluation notebook, so our local validation score matches the real leaderboard metric rather than an approximate proxy.
> Ahmadzadeh, A. *Self-Evaluation Notebook*, Solar Filament Segmentation Challenge 2026. https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook

In [ ]:
torch.cuda.empty_cache()  

In [ ]:
import torch
from pycocotools import mask as mask_util

def get_overlap_matrices(gt_layers, pred_layers):
    n_gt, height, width = gt_layers.shape
    n_pred = pred_layers.shape[0]
    gt_flat = gt_layers.reshape(n_gt, height * width)
    pred_flat = pred_layers.reshape(n_pred, height * width)
    intersection = torch.matmul(gt_flat, pred_flat.t())
    gt_areas = gt_flat.sum(dim=1).view(-1, 1)
    pred_areas = pred_flat.sum(dim=1).view(1, -1)
    union = gt_areas + pred_areas - intersection
    iou_matrix = torch.where(union == 0, torch.tensor(0., device=gt_layers.device), intersection / union)
    dice_matrix = torch.where(union == 0, torch.tensor(0., device=gt_layers.device), 2 * intersection / (gt_areas + pred_areas))
    return iou_matrix, dice_matrix

def fp_count_hit(hit_matrix):
    return (hit_matrix.sum(dim=0) == 0).sum().item()

def fn_count_hit(hit_matrix):
    return (hit_matrix.sum(dim=1) == 0).sum().item()

def rles_to_layers(rles, height=2048, width=2048):
    if not rles:
        return np.zeros((0, height, width), dtype=np.float32)
    rle_dicts = [{"size": [height, width], "counts": rle} for rle in rles]
    masks = mask_util.decode(rle_dicts)
    return masks.transpose(2, 0, 1).astype(np.float32)

def process_entry(annotator_image, gt_annotator_image_ids, pred_image_ids, gt_df, pred_df):
    image_id = annotator_image.split("-", maxsplit=1)[1]
    gt_rles = gt_df.loc[gt_annotator_image_ids == annotator_image, "segmentation_rle"].tolist()
    gt_layers = torch.from_numpy(rles_to_layers(gt_rles))
    pred_rles = pred_df.loc[pred_image_ids == image_id, "segmentation_rle"].tolist()
    pred_layers = torch.from_numpy(rles_to_layers(pred_rles))
    n_gt, n_pred = len(gt_rles), len(pred_rles)
    iou_matrix, dice_matrix = get_overlap_matrices(gt_layers, pred_layers)
    return iou_matrix, dice_matrix, n_gt, n_pred

def get_overlap_df(gt_df, pred_df):
    gt_annotator_image_ids = gt_df["filament_id"].str.split("_", n=1).str[0]
    pred_image_ids = pred_df["filament_id"].str.split("_", n=1).str[0]
    annotator_images = gt_annotator_image_ids.unique()
    overlap_df = pd.DataFrame({"annotator_image": annotator_images})
    results = overlap_df["annotator_image"].apply(process_entry, gt_annotator_image_ids=gt_annotator_image_ids,
                                                    pred_image_ids=pred_image_ids, gt_df=gt_df, pred_df=pred_df)
    overlap_df["iou_matrix"], overlap_df["dice_matrix"], overlap_df["n_gt"], overlap_df["n_pred"] = zip(*results)
    return overlap_df

def get_pq_score(overlap_df):
    iou_threshold = 0.5
    tp_iou_scores, fp_count, fn_count = [], 0, 0
    for row in overlap_df.itertuples(index=False):
        iou_matrix, n_gt, n_pred = row.iou_matrix, row.n_gt, row.n_pred
        if n_gt == 0:
            fp_count += n_pred; continue
        if n_pred == 0:
            fn_count += n_gt; continue
        hit_matrix = iou_matrix > iou_threshold
        tp_iou_scores.extend(iou_matrix[hit_matrix].tolist())
        fp_count += fp_count_hit(hit_matrix)
        fn_count += fn_count_hit(hit_matrix)
    denominator = len(tp_iou_scores) + 0.5 * fp_count + 0.5 * fn_count
    return sum(tp_iou_scores) / denominator if denominator > 0 else 0.0

def get_pq_breakdown(overlap_df):
    iou_threshold = 0.5
    tp_iou_scores, fp_count, fn_count = [], 0, 0
    for row in overlap_df.itertuples(index=False):
        iou_matrix, n_gt, n_pred = row.iou_matrix, row.n_gt, row.n_pred
        if n_gt == 0:
            fp_count += n_pred; continue
        if n_pred == 0:
            fn_count += n_gt; continue
        hit_matrix = iou_matrix > iou_threshold
        tp_iou_scores.extend(iou_matrix[hit_matrix].tolist())
        fp_count += fp_count_hit(hit_matrix)
        fn_count += fn_count_hit(hit_matrix)
    tp = len(tp_iou_scores)
    dq = tp / (tp + 0.5*fp_count + 0.5*fn_count) if (tp+fp_count+fn_count) > 0 else 0.0
    sq = sum(tp_iou_scores) / tp if tp > 0 else 0.0
    print(f"PQ: {dq*sq:.4f} | DQ (detection): {dq:.4f} | SQ (mask quality on matches): {sq:.4f}")
    print(f"TP: {tp} | FP: {fp_count} | FN: {fn_count}")
    return dq, sq

In [ ]:
def build_val_gt_df(coco, val_ids):
    rows = []
    for image_id in val_ids:
        for i, ann in enumerate(coco.loadAnns(coco.getAnnIds(imgIds=[image_id])), start=1):
            rows.append({'filament_id': f'{image_id}_{i}', 'segmentation_rle': mask_to_rle(coco.annToMask(ann))})
    return pd.DataFrame(rows)

def build_val_pred_df(coco, val_ids, img_dir, conf_threshold=0.25, mask_threshold=0.5):
    # several val_ids can be different annotators of the SAME photo — only run inference once per unique photo
    seen_photos = {}
    for image_id in val_ids:
        plain_id = image_id.split('-', 1)[1] if '-' in image_id else image_id
        seen_photos.setdefault(plain_id, image_id)

    rows = []
    for plain_id, rep_id in seen_photos.items():
        img_path = os.path.join(img_dir, coco.loadImgs([rep_id])[0]['file_name'])
        for i, mask in enumerate(predict_filaments_for_image(img_path, conf_threshold, mask_threshold=mask_threshold), start=1):
            if mask.sum() == 0:
                continue
            rows.append({'filament_id': f'{plain_id}_{i}', 'segmentation_rle': mask_to_rle(mask)})
    return pd.DataFrame(rows)

gt_df = build_val_gt_df(coco, val_ids)
pred_df = build_val_pred_df(coco, val_ids, TRAIN_IMG_DIR)

overlap_df = get_overlap_df(gt_df, pred_df)
print(f"Official PQ score on your validation set: {get_pq_score(overlap_df):.4f}")

get_pq_breakdown(overlap_df)

In [ ]:
best_conf, best_score = 0.25, 0.0
for conf_t in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35,0.40]:
    pred_df_t = build_val_pred_df(coco, val_ids, TRAIN_IMG_DIR, conf_threshold=conf_t)
    overlap_df_t = get_overlap_df(gt_df, pred_df_t)
    score = get_pq_score(overlap_df_t)
    print(f"conf={conf_t}: PQ={score:.4f}")
    if score > best_score:
        best_score, best_conf = score, conf_t
print(f"Best conf_threshold: {best_conf} (PQ={best_score:.4f})")

## 6. Building the Submission File

Predictions are formatted per the competition's required schema: one row per predicted filament (`filament_id`, `segmentation_rle`), masks encoded as COCO RLE via `pycocotools`.

In [ ]:
all_rows = []
FINAL_CONF_THRESHOLD = best_conf   # tuned against the real PQ metric in Section 6
FINAL_MASK_THRESHOLD = 0.5

for path in test_image_paths:
    image_id = os.path.splitext(os.path.basename(path))[0]
    for i, single_mask in enumerate(predict_filaments_for_image(path, conf_threshold=FINAL_CONF_THRESHOLD, mask_threshold=FINAL_MASK_THRESHOLD), start=1):
        if single_mask.sum() == 0:
            continue
        all_rows.append({'filament_id': f'{image_id}_{i}', 'segmentation_rle': mask_to_rle(single_mask)})

submission_df = pd.DataFrame(all_rows)
submission_df.to_csv('submission.csv', index=False)
print(f"Saved {len(submission_df)} filament rows to submission.csv")
submission_df.head()

## 8. Submission Sanity Check

We verify locally that no two predicted filament masks overlap for the same image, per the competition's explicit submission requirement.

In [ ]:
import pandas as pd
import numpy as np
from pycocotools import mask as mask_utils

submission_df = pd.read_csv('submission.csv')

def rle_to_mask(rle_string, shape=(2048, 2048)):
    rle = {'counts': rle_string.encode('utf-8'), 'size': list(shape)}
    return mask_utils.decode(rle)

image_groups = submission_df['filament_id'].str.rsplit('_', n=1).str[0]
overlap_found = False

for image_id, group in submission_df.groupby(image_groups):
    combined = np.zeros((2048, 2048), dtype=np.uint8)
    for _, row in group.iterrows():
        m = rle_to_mask(row['segmentation_rle'])
        if (combined & m).sum() > 0:
            print(f"OVERLAP at {row['filament_id']}")
            overlap_found = True
        combined |= m

print("Clean — no overlaps." if not overlap_found else "Overlaps found — see above.")